# scikit-learn TF-IDF 실습 (기초)

**환경:** Google Colab / Jupyter Notebook
**준비물:** 단원 2의 `cleaned_<엑셀파일명>.csv` (없어도 예제 리뷰로 진행 가능)

## 오늘 배울 것
단어 개수만 세면 `좋다`, `만족`처럼 **어느 리뷰에나 나오는 단어**가 가장 큰 값을 가집니다.
그런데 이런 단어는 리뷰끼리 **구별하는 데 도움이 되지 않습니다.**

**TF-IDF**는 단어마다 점수를 매겨서
- **이 리뷰에 많이 나오고** (TF)
- **다른 리뷰에는 잘 안 나오는** (IDF)

단어에 높은 점수를 줍니다.

## 실습 진행 과정
| 시간 | 단계 | 할 일 | 확인할 결과 |
|---|---|---|---|
| 0. 준비 | 데이터와 예제 리뷰 불러오기 | 리뷰 수 |
| 1. TF-IDF 개념 | TF, IDF 뜻 알아보기 | 공식 이해 |
| 2. TfidfVectorizer | 예제 리뷰로 점수 표 만들기, IDF 직접 계산 | 직접 계산 = scikit-learn |
| 3. 리뷰별 핵심어 | 점수가 가장 큰 단어 뽑기 | 리뷰별 핵심어 |
| 4. 실제 데이터 | 흔한 단어·드문 단어, ✏️ 리뷰 골라 키워드 보기 | 키워드 5개 |
| 5. 정리 | 한 줄 정리, 연습 문제 | — |

> **실행 방법:** 셀을 클릭하고 `Shift + Enter`. 반드시 **위에서부터 순서대로** 실행하세요.

---
## 0. 준비 (3분)
- `cleaned_<엑셀파일명>.csv`를 Colab 왼쪽 **📁 파일 창**에 올리면 그 데이터를 씁니다.
- 파일이 없어면 실행은 되지만 4장 실제데이터에서 결과치가 잘못 나옵니다^^.
- `mini`는 개념을 확인할 때 쓰는 **예제 리뷰 5개**입니다.

| 번호 | mini 리뷰 | 눈여겨볼 점 |
|---|---|---|
| 0 | 배송 빠르다 포장 꼼꼼하다 | 칭찬 리뷰 |
| 1 | 배송 빠르다 포장 별로 | 1번과 2번은 **단어는 같고 순서만 다름** |
| 2 | 포장 빠르다 배송 별로 | (뜻은 서로 다름) |
| 3 | 가격 저렴하다 품질 좋다 | 3번과 4번은 **뜻이 반대** |
| 4 | 가격 비싸다 품질 안 좋다 | (`좋다` vs `안 좋다`) |

- `cleaned_<.....>.csv`를 Colab 왼쪽 **📁 파일 창**에 올립니다.^^


In [1]:
# 파일 찾기 도구
import glob
# 표 다루기 도구
import pandas as pd

# 개념 확인용 예제 리뷰 5개 (단원 2처럼 단어가 나뉜 모양)
mini = [
    "배송 빠르다 포장 꼼꼼하다",
    "배송 빠르다 포장 별로",
    "포장 빠르다 배송 별로",
    "가격 저렴하다 품질 좋다",
    "가격 비싸다 품질 안 좋다",
]

# 'cleaned_'로 시작하는 CSV 파일 찾기
files = glob.glob("cleaned_*.csv")

# 파일이 있으면 '토큰문자열' 열을 사용, 없으면 예제 리뷰를 반복해서 사용
if files:
    docs = pd.read_csv(files[0])["토큰문자열"].dropna().astype(str).tolist()
    print("데이터 파일:", files[0])
else:
    docs = mini * 20
    print("[안내] CSV 파일이 없어 예제 리뷰로 진행합니다.")
print("리뷰 수:", len(docs))

# 공통 설정: 띄어쓰기로만 단어 나누기, 영어 대소문자 그대로 두기
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

데이터 파일: cleaned_customer_inquiries.csv
리뷰 수: 3001


> 💡 **`opts`는 무엇인가요?** 단원 2에서 이미 단어를 나눴으니 "띄어쓰기로만 나눠 주세요"라고 알려 주는 설정입니다.
> 이 설정이 없으면 scikit-learn이 **한 글자 단어(`안`, `향`)를 지워** 버립니다. 뒤에서 `**opts`라고 쓰면 이 설정을 그대로 넣는다는 뜻입니다.

---
## 1. TF-IDF 개념

### 💡 비유: 반장 선거 공약
후보 모두가 "열심히 하겠습니다"라고 하면 이 말로는 후보를 **구별할 수 없습니다.**
한 후보만 "급식 메뉴 투표제"를 말했다면, 그것이 **그 후보만의 특징**입니다.

| 이름 | 뜻 | 커지는 경우 |
|---|---|---|
| **TF** (단어 빈도) | 이 리뷰에 몇 번 나왔나 | 이 리뷰에 많이 나올수록 |
| **IDF** (역문서 빈도) | 얼마나 **드문** 단어인가 | **적은 리뷰에만** 나올수록 |
| **TF-IDF** | TF × IDF | 이 리뷰에 많고, 다른 리뷰에는 드물수록 |

### scikit-learn의 IDF 공식 (기본 설정)
```
IDF = ln( (1 + 전체 리뷰 수) / (1 + 그 단어가 나온 리뷰 수) ) + 1
```
- 많은 리뷰에 나온 단어 → 분모가 큼 → **IDF가 작아짐**
- 적은 리뷰에 나온 단어 → 분모가 작음 → **IDF가 커짐**
- 마지막에 리뷰마다 숫자 목록의 길이를 1로 맞추는 **정규화**를 합니다.

> ⚠️ **교재마다 공식이 조금씩 다릅니다.** `+1`이 없는 공식도 많아서, 교재 공식으로 계산하면 scikit-learn 결과와 숫자가 다를 수 있습니다. 이 실습은 scikit-learn 공식을 따릅니다.

---
## 2. TfidfVectorizer로 점수 매기기 (7분)

### 2-1. TF-IDF 점수 표

In [2]:
# TF-IDF 도구
from sklearn.feature_extraction.text import TfidfVectorizer

# 개념 확인용 예제 리뷰 5개 (단원 2처럼 단어가 나뉜 모양)
mini = [
    "배송 빠르다 포장 꼼꼼하다",
    "배송 빠르다 포장 별로",
    "포장 빠르다 배송 별로",
    "가격 저렴하다 품질 좋다",
    "가격 비싸다 품질 안 좋다",
]
# 공통 설정: 띄어쓰기로만 단어 나누기, 영어 대소문자 그대로 두기
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False)

# 예제 리뷰로 TF-IDF 학습하고 점수 구하기
tv = TfidfVectorizer(**opts)
X = tv.fit_transform(mini)
words = tv.get_feature_names_out()

# 리뷰 × 단어 점수 표 (소수 둘째 자리까지)
pd.DataFrame(X.toarray(), columns=words, index=mini).round(2)

,가격,꼼꼼하다,배송,별로,비싸다,빠르다,안,저렴하다,좋다,포장,품질
배송 빠르다 포장 꼼꼼하다,0.00,0.65,0.44,0.00,0.0,0.44,0.0,0.00,0.00,0.44,0.00
배송 빠르다 포장 별로,0.00,0.00,0.47,0.57,0.0,0.47,0.0,0.00,0.00,0.47,0.00
포장 빠르다 배송 별로,0.00,0.00,0.47,0.57,0.0,0.47,0.0,0.00,0.00,0.47,0.00
가격 저렴하다 품질 좋다,0.47,0.00,0.00,0.00,0.0,0.00,0.0,0.58,0.47,0.00,0.47
가격 비싸다 품질 안 좋다,0.41,0.00,0.00,0.00,0.5,0.00,0.5,0.00,0.41,0.00,0.41


### 2-2. IDF 직접 계산해서 비교하기

In [4]:
# 수학 계산 도구 (ln 계산용)
import numpy as np

# 전체 리뷰 수
n = len(mini)

# 단어마다 '몇 개 리뷰에 나왔나' 세기
df = []
for w in words:
    count = 0
    for review in mini:
        if w in review.split():   # 리뷰에 그 단어가 있으면
            count += 1            # 1 더하기
    df.append(count)
df = np.array(df)

# 공식대로 직접 계산한 IDF
my_idf = np.log((1 + n) / (1 + df)) + 1

# 직접 계산한 값과 scikit-learn 값을 나란히 보기
pd.DataFrame({"나온 리뷰 수": df, "직접 계산": my_idf, "scikit-learn": tv.idf_}, index=words).round(3)

,나온 리뷰 수,직접 계산,scikit-learn
가격,2,1.693,1.693
꼼꼼하다,1,2.099,2.099
배송,3,1.405,1.405
별로,2,1.693,1.693
비싸다,1,2.099,2.099
빠르다,3,1.405,1.405
안,1,2.099,2.099
저렴하다,1,2.099,2.099
좋다,2,1.693,1.693
포장,3,1.405,1.405


✅ 두 열이 **똑같습니다.**
✅ 리뷰 3개에 나온 `배송`·`포장`은 IDF가 **낮고**, 리뷰 1개에만 나온 `꼼꼼하다`·`안`은 **높습니다.**

---
## 3. 리뷰별 핵심어 (5분)
리뷰마다 **점수가 가장 큰 단어**가 그 리뷰의 핵심어입니다.

In [5]:
# 리뷰마다 점수 상위 2개 단어 출력
for i in range(len(mini)):
    scores = X[i].toarray()[0]            # i번째 리뷰의 점수 목록
    top2 = scores.argsort()[::-1][:2]     # 점수가 큰 순서로 칸 번호 2개
    print(mini[i], "→", list(words[top2]))

배송 빠르다 포장 꼼꼼하다 → ['꼼꼼하다', '빠르다']
배송 빠르다 포장 별로 → ['별로', '빠르다']
포장 빠르다 배송 별로 → ['별로', '빠르다']
가격 저렴하다 품질 좋다 → ['저렴하다', '좋다']
가격 비싸다 품질 안 좋다 → ['안', '비싸다']


✅ 4번 리뷰의 핵심어에 **`안`** 이 나옵니다. 다른 리뷰에는 없는 단어라 점수가 높기 때문입니다.

---
## 4. 실제 데이터

### 4-1. 흔한 단어 vs 드문 단어

In [6]:
# 실제 데이터로 TF-IDF 학습 (5개 미만 리뷰에만 나온 단어는 제외)
tv_real = TfidfVectorizer(min_df=5, **opts)
X_real = tv_real.fit_transform(docs) #docs 는 엑셀파일
real_words = tv_real.get_feature_names_out()

# 단어별 IDF를 작은 순서로 정렬
idf = pd.Series(tv_real.idf_, index=real_words).sort_values()

# IDF가 낮은(흔한) 단어 10개와 높은(드문) 단어 10개
print("흔한 단어:", list(idf.head(10).index))
print("드문 단어:", list(idf.tail(10).index))

흔한 단어: ['좋다', '사용', '제품', '향', '같다', '쓰다', '않다', '구매', '가격', '바르다']
드문 단어: ['단계', '다행', '나라', '꺼리다', '기본', '국산', '교체', '계속', '결과', '거칠다']


### 4-2. ✏️ 리뷰 하나의 키워드 보기
`review_no`를 바꿔 여러 리뷰의 키워드를 확인해 보세요.

In [7]:
# ===== ✏️ [입력] 볼 리뷰 번호 =====
review_no = 0
# ================================

# 그 리뷰의 단어와 점수 출력
print("리뷰 단어:", docs[review_no])
scores = X_real[review_no].toarray()[0]

# 점수가 큰 단어 5개 (점수 0은 제외)
top5 = [i for i in scores.argsort()[::-1][:5] if scores[i] > 0]
for i in top5:
    print(f"  {real_words[i]}: {scores[i]:.2f}")

리뷰 단어: 남편 사다 광고 말 향 않다 올드 남자 스킨 로션 향 향
  올드: 0.49
  향: 0.40
  말: 0.40
  광고: 0.39
  남자: 0.30


🤔 **생각해 볼 점**
- 단원 2에서 불용어를 이미 뺐지만, IDF도 흔한 단어의 점수를 **자동으로** 낮춥니다. 그래서 "TF-IDF를 쓸 때 불용어를 꼭 빼야 할까?"는 **의견이 갈리는 문제**입니다.
- 리뷰가 아주 짧으면(단어 2~3개) 점수가 한두 단어에 몰립니다. 짧은 글에서는 "나왔나/안 나왔나"만 보는 방법이 더 안정적일 수 있다고 scikit-learn 공식 문서에 적혀 있습니다.

---
## 5. 정리 (3분)

### ✅ 한 줄 정리
**TF-IDF = 이 리뷰에 많고, 다른 리뷰에는 드문 단어에 높은 점수.** 리뷰별 핵심 키워드를 뽑을 수 있습니다.

| 방법 | 흔한 단어 `좋다` | 드문 단어 `꼼꼼하다` |
|---|---|---|
| 단어 개수 (BoW) | 큰 값 | 작은 값 |
| TF-IDF | **작은 점수** | **큰 점수** |





**참고 공식 문서:** [scikit-learn TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) · [텍스트 특징 추출](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction)